# Notebook 02: Neutrophil Dataset Parsing

**Input**: Cuffdiff output (GSE40548) + DESeq2 output (GSE221866)
**Output**: Parsed gene lists with log2FC and FDR for both neutrophil datasets

| Dataset | Stimulus | Platform | DE Method | Samples |
|---------|----------|----------|-----------|---------|
| GSE40548 | TNF-alpha (10 ng/mL, 1h) | Illumina HiSeq + SOLiD | Cuffdiff | n=3 |
| GSE221866 | LPS (1 ug/mL, 6h) | NovaSeq PE100 | DESeq2 | n=3 |

In [ ]:
import pandas as pd
import numpy as np
import os

OUTPUT_DIR = '../tables'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Notebook 02: Neutrophil Dataset Parsing')

## 2.1 Parse GSE40548 — Cuffdiff (TNF-alpha neutrophils)

In [ ]:
# GSE40548: Cuffdiff output
# Source: Wright et al., 2013, PLoS One
# Columns: gene_id, gene, locus, sample_1, sample_2, status, value_1, value_2,
#          log2(fold_change), test_stat, p_value, q_value, significant

gse40548 = pd.read_csv('/mnt/agents/output/data/gse40548_cuffdiff.txt', sep='\t')

# Standardize column names
gse40548 = gse40548.rename(columns={
    'gene': 'gene_symbol',
    'log2(fold_change)': 'log2FC',
    'q_value': 'FDR'
})

# Remove rows with NA values
gse40548 = gse40548.dropna(subset=['log2FC', 'FDR'])

# Filter for protein-coding genes (remove non-informative entries)
gse40548 = gse40548[gse40548['gene_symbol'].str.len() > 1]
gse40548 = gse40548[~gse40548['gene_symbol'].str.startswith('MIR')]

# Keep one entry per gene (largest |log2FC|)
gse40548['abs_log2FC'] = gse40548['log2FC'].abs()
gse40548 = gse40548.sort_values('abs_log2FC', ascending=False)
gse40548 = gse40548.drop_duplicates(subset=['gene_symbol'], keep='first')

# Summary
print(f'GSE40548 (TNF-alpha neutrophils):')
print(f'  Total genes: {len(gse40548):,}')
print(f'  |log2FC| > 1: {(gse40548["abs_log2FC"] > 1).sum():,}')
print(f'  |log2FC| > 1 & FDR < 0.05: {((gse40548["abs_log2FC"] > 1) & (gse40548["FDR"] < 0.05)).sum():,}')
print(f'  Upregulated (FC>1, FDR<0.05): {((gse40548["log2FC"] > 1) & (gse40548["FDR"] < 0.05)).sum():,}')
print(f'  Downregulated (FC<-1, FDR<0.05): {((gse40548["log2FC"] < -1) & (gse40548["FDR"] < 0.05)).sum():,}')

# Top genes
print(f'\nTop 5 upregulated:')
print(gse40548[gse40548['log2FC'] > 0].head(5)[['gene_symbol', 'log2FC', 'FDR']].to_string(index=False))
print(f'\nTop 5 downregulated:')
print(gse40548[gse40548['log2FC'] < 0].head(5)[['gene_symbol', 'log2FC', 'FDR']].to_string(index=False))

# Save
gse40548[['gene_symbol', 'log2FC', 'FDR']].to_csv(
    f'{OUTPUT_DIR}/Neut_TNF_GSE40548_differential_expression.csv', index=False
)
print(f'\nSaved: Neut_TNF_GSE40548_differential_expression.csv')

## 2.2 Parse GSE221866 — DESeq2 (LPS neutrophils)

In [ ]:
# GSE221866: DESeq2 output from supplementary Excel
# Source: Ismailova et al., 2023 (Front Immunol, DOI: 10.3389/fimmu.2023.1123344)
# Columns: gene_symbol, baseMean, log2FoldChange, lfcSE, stat, pvalue, padj

gse221866 = pd.read_excel('../data/GSE221866_DESeq2.xlsx')

# Standardize column names
gse221866 = gse221866.rename(columns={
    'log2FoldChange': 'log2FC',
    'padj': 'FDR'
})

# Remove rows with NA
gse221866 = gse221866.dropna(subset=['log2FC', 'FDR'])

# Filter protein-coding
gse221866 = gse221866[gse221866['gene_symbol'].str.len() > 1]
gse221866 = gse221866[~gse221866['gene_symbol'].str.startswith('MIR')]

# Deduplicate
gse221866['abs_log2FC'] = gse221866['log2FC'].abs()
gse221866 = gse221866.sort_values('abs_log2FC', ascending=False)
gse221866 = gse221866.drop_duplicates(subset=['gene_symbol'], keep='first')

# Summary
print(f'GSE221866 (LPS neutrophils):')
print(f'  Total genes: {len(gse221866):,}')
print(f'  |log2FC| > 1: {(gse221866["abs_log2FC"] > 1).sum():,}')
print(f'  |log2FC| > 1 & FDR < 0.05: {((gse221866["abs_log2FC"] > 1) & (gse221866["FDR"] < 0.05)).sum():,}')
print(f'  Upregulated: {((gse221866["log2FC"] > 1) & (gse221866["FDR"] < 0.05)).sum():,}')
print(f'  Downregulated: {((gse221866["log2FC"] < -1) & (gse221866["FDR"] < 0.05)).sum():,}')

# Top genes
print(f'\nTop 5 upregulated:')
print(gse221866[gse221866['log2FC'] > 0].head(5)[['gene_symbol', 'log2FC', 'FDR']].to_string(index=False))
print(f'\nTop 5 downregulated:')
print(gse221866[gse221866['log2FC'] < 0].head(5)[['gene_symbol', 'log2FC', 'FDR']].to_string(index=False))

# Save
gse221866[['gene_symbol', 'log2FC', 'FDR']].to_csv(
    f'{OUTPUT_DIR}/Neut_LPS_GSE221866_differential_expression.csv', index=False
)
print(f'\nSaved: Neut_LPS_GSE221866_differential_expression.csv')

## 2.3 Intersection and Common Gene Space

In [ ]:
# Load NK results from Notebook 01
nk_de = pd.read_csv(f'{OUTPUT_DIR}/NK_IL2_GSE50838_differential_expression.csv')
nk_genes = set(nk_de['gene'])

tnf_genes = set(gse40548['gene_symbol'])
lps_genes = set(gse221866['gene_symbol'])

# Three-way intersection
common_genes = nk_genes & tnf_genes & lps_genes

print(f'Common gene space across all 3 datasets: {len(common_genes):,} genes')
print(f'  NK (GSE50838): {len(nk_genes):,}')
print(f'  Neut TNF (GSE40548): {len(tnf_genes):,}')
print(f'  Neut LPS (GSE221866): {len(lps_genes):,}')
print(f'  NK ∩ TNF: {len(nk_genes & tnf_genes):,}')
print(f'  NK ∩ LPS: {len(nk_genes & lps_genes):,}')
print(f'  TNF ∩ LPS: {len(tnf_genes & lps_genes):,}')
print(f'  NK ∩ TNF ∩ LPS: {len(common_genes):,}')